# Bank Marketing — Customer Subscription Prediction

## 1. Problem Definition & Project Objective

### Problem Statement

The objective of this project is to predict whether a bank
customer will subscribe to a term deposit based on customer
and marketing campaign information.

### Machine Learning Objective

This is a supervised binary classification problem where the
target variable `y` indicates whether the customer subscribed
to a term deposit.

### Business Objective

The model aims to identify customers who are more likely to
subscribe to a term deposit, potentially enabling more
targeted marketing campaigns.

## 2. Import Libraries

We use Pandas to load and inspect the dataset and NumPy for numerical operations that may be needed during analysis.

In [1]:
import pandas as pd
import numpy as np 

## 3. Load Dataset

The Bank Marketing dataset contains customer, financial, and marketing campaign information. We use `bank-full.csv` as our primary dataset because it contains 45,211 records.

The file uses a semicolon (`;`) as its field separator.

In [4]:
df = pd.read_csv("../data/bank-full.csv", sep=";")

df.head()


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 4. Data Understanding

Before performing data cleaning and exploratory data analysis, we inspect the dataset's structure, variables, data types, and basic quality.

### 4.1 Dataset Shape

We inspect the number of records and columns to understand the size of our dataset.

In [5]:
df.shape

(45211, 17)

The dataset contains 45,211 rows and 17 columns. These columns consist of 16 input features and one target variable.

### 4.2 Dataset Information

We inspect the column names, non-null counts, data types, and memory usage.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 5.9 MB


The dataset contains numerical and categorical variables. The non-null counts help us identify columns with missing values, while the data types guide our later preprocessing decisions.

### 4.3 Column Names

We list the column names to identify the available input variables and target.

In [7]:
df.columns.tolist()

['age',
 'job',
 'marital',
 'education',
 'default',
 'balance',
 'housing',
 'loan',
 'contact',
 'day',
 'month',
 'duration',
 'campaign',
 'pdays',
 'previous',
 'poutcome',
 'y']

The dataset contains customer attributes, account information, contact details, and campaign history. The `y` column is our target variable.

### 4.4 Numerical and Categorical Features

We identify the numerical and categorical input features so that suitable preprocessing can be applied later.

In [8]:
numerical_features = [
    "age", "balance", "day", "duration",
    "campaign", "pdays", "previous"
]

categorical_features = [
    "job", "marital", "education", "default",
    "housing", "loan", "contact", "month", "poutcome"
]

target = "y"

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Target:", target)

Numerical features: 7
Categorical features: 9
Target: y


The dataset contains seven numerical features and nine categorical features. The target `y` indicates whether a customer subscribed to the term deposit. We will use these feature groups when building the preprocessing pipeline.

### 4.5 Missing Values

We check each column for missing values before deciding whether any imputation is required.

In [9]:
df.isnull().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

The initial inspection shows no missing values in the dataset. Therefore, missing-value imputation is not required based on this check.

### 4.6 Duplicate Records

We check for completely duplicated rows to determine whether exact duplicate records are present.

In [10]:
df.duplicated().sum()

np.int64(0)

No completely duplicated rows were found. Therefore, duplicate-row removal is not required based on this check.

### 4.7 Target Distribution

We inspect the number and percentage of records belonging to each target class to understand the distribution of subscription outcomes.

In [11]:
target_counts = df[target].value_counts()

print("Target counts:")
print(target_counts)

print("\nTarget percentages:")
print((df[target].value_counts(normalize=True) * 100).round(2))

Target counts:
y
no     39922
yes     5289
Name: count, dtype: int64

Target percentages:
y
no     88.3
yes    11.7
Name: proportion, dtype: float64


The target is imbalanced because non-subscriptions are substantially more frequent than subscriptions. We will consider this when evaluating models and will not rely on accuracy alone.

### 4.8 Descriptive Statistics

We summarize the numerical features to obtain an initial view of their ranges, central values, and variability.

In [12]:
df.describe()

,age,balance,day,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


The summary provides statistics such as the mean, standard deviation, quartiles, minimum, and maximum. We will investigate distributions and unusual values more closely during EDA.

### 4.9 Unique Values

We count the distinct values in each column to understand the variety of values present, particularly in categorical features.

In [13]:
df.nunique()

age            77
job            12
marital         3
education       4
default         2
balance      7168
housing         2
loan            2
contact         3
day            31
month          12
duration     1573
campaign       48
pdays         559
previous       41
poutcome        4
y               2
dtype: int64

The number of unique values helps us understand categorical feature cardinality and plan preprocessing. Unique-value counts alone do not determine which encoding technique should be used.

### 4.10 Important Observations

The initial inspection establishes the following:

- The primary dataset contains 45,211 records and 17 columns.
- It has 16 input features and one target variable, `y`.
- The input features include numerical and categorical variables.
- No missing values or completely duplicated rows were found.
- The target classes are imbalanced, with approximately 88.3% `no` and 11.7% `yes`.
- The `pdays` feature uses `-1` to represent customers who were not previously contacted.

These findings guide the next stages of data cleaning and exploratory data analysis.